# ALLdis GBD 2010—2023更新后联合PSA烟雾测试审计 v1.3

本Notebook记录2批×100次技术烟雾测试。所有区间仅用于验证引擎，不能作为论文正式概率性结果。

In [1]:
from pathlib import Path
import json
import pandas as pd

HERE = Path.cwd().resolve()
ROOT = HERE if (HERE / 'input' / 'psa').exists() else HERE.parent
MERGED = ROOT / 'psa_smoke_recheck' / 'merged'
print('Bundle root:', ROOT)

Bundle root: /workspace/scratch/a723b9488ae5/outputs/ALLdis_GBD2010_2023更新与正式PSA候选_v1.3


## 1. 自动质量控制

In [2]:
manifest = json.loads((MERGED / 'MERGED_MANIFEST.json').read_text(encoding='utf-8'))
qc = pd.read_csv(MERGED / '00_QC.csv')
print(manifest)
display(qc)
assert manifest['formal_psa_run'] is False
assert (qc['status'] == 'PASS').all()

{"status": "PASS", "formal_psa_run": false, "draws": 200, "batches": 2, "cea_rows": 4800, "qc_pass": 20, "qc_total": 20, "interpretation": "Technical smoke test only; do not report probabilistic intervals as formal results"}


check_id,status,observed,expected,note
DRAW_COUNT,PASS,200,200,NaN
DRAW_ID_UNIQUE_ACROSS_BATCHES,PASS,200,200,NaN
CEA_ROW_COUNT,PASS,4800,4800,NaN
SCENARIO_COUNT,PASS,8,8,NaN
HORIZON_SET,PASS,"[np.int64(5), np.int64(10), np.int64(26)]","[5, 10, 26]",NaN
S0_HEALTH_ZERO,PASS,0.0,0,NaN
S0_COST_ZERO,PASS,0.0,0,NaN
S7_COST_MISSING,PASS,0,0,NaN
ELIGIBLE_HEALTH_COMPLETE,PASS,0,0,NaN
ELIGIBLE_COST_COMPLETE,PASS,0,0,NaN


## 2. 与确定性结果衔接

In [3]:
recon = pd.read_csv(MERGED / '06_deterministic_reconciliation.csv')
display(recon)
assert (recon['qc_status'] == 'PASS').all()

horizon_years,scenario_id,metric,psa_smoke_mean,deterministic_value,mean_to_deterministic_ratio,relative_difference,tolerance,qc_status
5,S6,discounted_dalys_averted,1.077523e+06,1.090808e+06,0.987821,-0.012179,0.15,PASS
5,S6,undiscounted_incidence_averted,1.026213e+06,1.003912e+06,1.022215,0.022215,0.15,PASS
5,S6,discounted_programme_cost_2025_cny,1.630018e+10,1.648440e+10,0.988825,-0.011175,0.15,PASS
10,S6,discounted_dalys_averted,6.148624e+06,6.245588e+06,0.984475,-0.015525,0.15,PASS
10,S6,undiscounted_incidence_averted,4.822522e+06,4.737474e+06,1.017952,0.017952,0.15,PASS
10,S6,discounted_programme_cost_2025_cny,2.567277e+10,2.596291e+10,0.988825,-0.011175,0.15,PASS
26,S6,discounted_dalys_averted,3.051860e+07,2.970895e+07,1.027253,0.027253,0.15,PASS
26,S6,undiscounted_incidence_averted,2.498524e+07,2.368487e+07,1.054903,0.054903,0.15,PASS
26,S6,discounted_programme_cost_2025_cny,4.784751e+10,4.838826e+10,0.988825,-0.011175,0.15,PASS


## 3. S6技术性区间

In [4]:
summary = pd.read_csv(MERGED / '02_PSA_summary.csv')
metrics = ['discounted_dalys_averted','discounted_programme_cost_2025_cny','discounted_partial_public_payer_net_cost_2025_cny']
s6 = summary[(summary.scenario_id == 'S6') & summary.metric.isin(metrics)]
display(s6[['horizon_years','metric','mean','p2_5','median','p97_5','result_status']])

horizon_years,metric,mean,p2_5,median,p97_5,result_status
5,discounted_dalys_averted,1.077523e+06,6.297997e+05,1.067245e+06,1.613610e+06,SMOKE_ONLY_NOT_FORMAL_PSA
5,discounted_programme_cost_2025_cny,1.630018e+10,8.731535e+09,1.598954e+10,2.633807e+10,SMOKE_ONLY_NOT_FORMAL_PSA
5,discounted_partial_public_payer_net_cost_2025_cny,1.481304e+10,7.607759e+09,1.462459e+10,2.483215e+10,SMOKE_ONLY_NOT_FORMAL_PSA
10,discounted_dalys_averted,6.148624e+06,3.647818e+06,6.069144e+06,9.201743e+06,SMOKE_ONLY_NOT_FORMAL_PSA
10,discounted_programme_cost_2025_cny,2.567277e+10,1.375216e+10,2.518351e+10,4.148243e+10,SMOKE_ONLY_NOT_FORMAL_PSA
10,discounted_partial_public_payer_net_cost_2025_cny,1.613918e+10,-8.203369e+08,1.587700e+10,3.024411e+10,SMOKE_ONLY_NOT_FORMAL_PSA
26,discounted_dalys_averted,3.051860e+07,1.678930e+07,2.990010e+07,4.788743e+07,SMOKE_ONLY_NOT_FORMAL_PSA
26,discounted_programme_cost_2025_cny,4.784751e+10,2.563053e+10,4.693566e+10,7.731270e+10,SMOKE_ONLY_NOT_FORMAL_PSA
26,discounted_partial_public_payer_net_cost_2025_cny,2.105426e+09,-5.814883e+10,5.103051e+09,3.601008e+10,SMOKE_ONLY_NOT_FORMAL_PSA


## 4. 结论

GBD趋势、政策效果、成本、RR、身体活动基线、D4费用和D5支付比例均已进入抽样；20/20项烟雾测试QC通过后，可使用独立的正式运行Notebook执行10,000次PSA。